# [RQ_2 · HỖ TRỢ] Config B — Tinh chỉnh Random Forest

## Mục tiêu

Tìm bộ tham số Random Forest có MAE trung bình thấp nhất qua 3 fold theo thời gian trên tập train. Giữ 27 đặc trưng thời gian, ngày lễ và thời tiết như thí nghiệm ngày 13/11.

Mốc đối chiếu ngày 13/11 là MAE CV 281,28 và MAE validation 240,10 xe/giờ, với 400 cây, độ sâu 12, tối thiểu 5 mẫu mỗi lá và `max_features=1.0`.

- Lượt 1 thử độ sâu và số mẫu mỗi lá, giữ nguyên số cây và mặc định dùng toàn bộ đặc trưng ở mỗi nút.
- Lượt 2 là tùy chọn, thử tỉ lệ đặc trưng ở mỗi nút cho hai bộ đứng đầu lượt 1.
- Validation dùng bộ được chọn theo CV. Bật RUN_VALIDATION sau khi xem bảng CV.

**Nguồn dữ liệu chung (19/11/2026):** Code đọc dữ liệu đã chuyển sang `src.experiment_data.load_experiment_splits`. `processed` dùng snapshot bàn giao; `raw` dùng cho CV trước điền thiếu; `filled` giữ schema nhãn và học điền từ train. Các output/metrics đang hiển thị là kết quả lịch sử, chưa chạy lại sau thay đổi này.


## Params

In [12]:
EXPERIMENT_NAME = "16-11-2026_ConfigB_RandomForest_Phuc"
CONFIG_NAME = "B"
MODEL_NAME = "Random Forest"
TARGET = "traffic_volume"
RANDOM_STATE = 42
CV_SPLITS = 3
RUN_MAX_FEATURES_CV = False
RUN_VALIDATION = False

DATA_PARAMS = {"mask_stuck_temp": False, "complete_holidays": False}

REFERENCE_PARAMS = {
    "n_estimators": 400,
    "max_depth": 12,
    "min_samples_leaf": 5,
    "max_features": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

PARAM_GRIDS = {
    MODEL_NAME: {
        "max_depth": [8, 10, 12, 14],
        "min_samples_leaf": [3, 5, 8, 12],
    },
}
MAX_FEATURES_TO_TRY = [0.6, 0.8]
REFERENCE_VALIDATION_MAE = 240.09808540736057

## Import và tìm thư mục dự án

In [13]:
import json
import sys
import warnings
from pathlib import Path
from datetime import datetime, timezone, timedelta
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root():
    current = Path.cwd().resolve()

    for folder in (current, *current.parents):
        for candidate in (
            folder,
            folder / "metro_interstate_traffic_volume",
        ):
            if (
                (candidate / "src" / "models.py").is_file()
                and (candidate / "src" / "features.py").is_file()
            ):
                return candidate

    raise FileNotFoundError(
        "Không tìm thấy dự án. Hãy kiểm tra thư mục làm việc của kernel."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import features
from src.evaluation import clip_prediction, regression_metrics

DATA_PATH = PROJECT_ROOT / "data" / "Metro_Interstate_Traffic_Volume.csv"

RUN_ID = datetime.now(
    timezone(timedelta(hours=7))
).strftime("%Y%m%d_%H%M%S_%f")

OUTPUT_DIR = (
    PROJECT_ROOT
    / "experiments"
    / EXPERIMENT_NAME
    / "outputs"
    / RUN_ID
)

print("Project:", PROJECT_ROOT)
print("Data:", DATA_PATH)
print("Output:", OUTPUT_DIR)

assert DATA_PATH.is_file(), f"Không tìm thấy dữ liệu: {DATA_PATH}"
REFERENCE_DIR = PROJECT_ROOT / "experiments" / "13-11-2026_ConfigB_4Model_Phuc"
# Nạp helper bằng đường dẫn cụ thể khi kernel được mở từ thư mục khác.
import importlib.util
spec = importlib.util.spec_from_file_location(
    "config_b_grid_cv", REFERENCE_DIR / "config_b_grid_cv.py"
)
grid_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(grid_module)
make_config_b_pipeline = grid_module.make_config_b_pipeline
run_grid_cv = grid_module.run_grid_cv

Project: D:\SGU_HocMay_2026-2027\metro_interstate_traffic_volume
Data: D:\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\data\Metro_Interstate_Traffic_Volume.csv
Output: D:\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\experiments\16-11-2026_ConfigB_RandomForest_Phuc\outputs\20261116_214515_288550


## Đọc dữ liệu train và kiểm tra Config B

In [14]:
FEATURE_COLUMNS = list(features.CONFIG_B)
from src.experiment_data import load_experiment_splits
raw_train = load_experiment_splits(path=DATA_PATH, **DATA_PARAMS, stage="raw")["train"]

assert len(FEATURE_COLUMNS) == 27
assert TARGET not in FEATURE_COLUMNS
assert set(FEATURE_COLUMNS).isdisjoint(features.HISTORY_FEATURES)
assert raw_train["date_time"].is_monotonic_increasing
assert not raw_train["date_time"].duplicated().any()

display(pd.DataFrame([{
    "split": "train",
    "rows": len(raw_train),
    "start": raw_train["date_time"].min(),
    "end": raw_train["date_time"].max(),
    "n_features": len(FEATURE_COLUMNS),
}]))
print("Features:", FEATURE_COLUMNS)

,split,rows,start,end,n_features
0,train,28402,2012-10-02 09:00:00,2017-05-10 04:00:00,27


Features: ['hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'is_holiday', 'temp_c', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_clear', 'weather_clouds', 'weather_drizzle', 'weather_fog', 'weather_haze', 'weather_mist', 'weather_rain', 'weather_smoke', 'weather_snow', 'weather_squall', 'weather_thunderstorm']


## Lượt 1 · Grid CV theo thời gian

Gọi lại helper của thí nghiệm ngày 13/11 để xử lý thời tiết, tạo đặc trưng và tính MAE giống lần trước. Lưới gồm 16 tổ hợp và 48 lần fit, kể cả bộ đối chiếu. Nếu bộ dẫn đầu nằm ở biên lưới, có thể mở rộng đúng phía đó trong `PARAM_GRIDS` trước khi thử lượt 2.

In [15]:
cv_depth_leaf = run_grid_cv(
    raw_train, {MODEL_NAME: REFERENCE_PARAMS}, PARAM_GRIDS, n_splits=CV_SPLITS,
).assign(stage="depth_leaf")

if not any(params == REFERENCE_PARAMS for params in cv_depth_leaf["params"]):
    reference_cv = run_grid_cv(
        raw_train, {MODEL_NAME: REFERENCE_PARAMS}, {MODEL_NAME: {}}, n_splits=CV_SPLITS,
    ).assign(stage="reference")
    cv_depth_leaf = pd.concat([cv_depth_leaf, reference_cv], ignore_index=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cv_depth_leaf.to_csv(OUTPUT_DIR / "grid_cv_depth_leaf.csv", index=False)

display(cv_depth_leaf.sort_values("mean_MAE", kind="stable").reset_index(drop=True))

Random Forest: bộ 1/16
Random Forest: bộ 2/16
Random Forest: bộ 3/16
Random Forest: bộ 4/16
Random Forest: bộ 5/16
Random Forest: bộ 6/16
Random Forest: bộ 7/16
Random Forest: bộ 8/16
Random Forest: bộ 9/16
Random Forest: bộ 10/16
Random Forest: bộ 11/16
Random Forest: bộ 12/16
Random Forest: bộ 13/16
Random Forest: bộ 14/16
Random Forest: bộ 15/16
Random Forest: bộ 16/16


,model,candidate_id,params,fold_1_MAE,fold_1_RMSE,fold_1_R2,fold_2_MAE,fold_2_RMSE,fold_2_R2,fold_3_MAE,fold_3_RMSE,fold_3_R2,mean_MAE,std_MAE,mean_RMSE,std_RMSE,mean_R2,std_R2,warnings,stage
0,Random Forest,10,"{'n_estimators': 400, 'max_depth': 12, 'min_sa...",266.830703,473.474793,0.943610,265.303115,432.351205,0.953289,311.718133,495.621849,0.934363,281.283984,21.529228,467.149282,26.214533,0.943754,0.007727,,depth_leaf
1,Random Forest,6,"{'n_estimators': 400, 'max_depth': 10, 'min_sa...",269.215910,474.390300,0.943391,265.003368,431.665157,0.953437,310.166765,494.298692,0.934713,281.462014,20.370050,466.784716,26.129467,0.943847,0.007651,,depth_leaf
2,Random Forest,5,"{'n_estimators': 400, 'max_depth': 10, 'min_sa...",270.215255,473.645372,0.943569,264.961597,432.312283,0.953297,310.298895,494.878262,0.934560,281.825249,20.247825,466.945306,25.978114,0.943809,0.007651,,depth_leaf
3,Random Forest,11,"{'n_estimators': 400, 'max_depth': 12, 'min_sa...",267.993973,474.336461,0.943404,266.780411,435.121220,0.952688,311.465486,495.290673,0.934451,282.079957,20.784613,468.249451,24.938318,0.943515,0.007446,,depth_leaf
4,Random Forest,14,"{'n_estimators': 400, 'max_depth': 14, 'min_sa...",267.187081,473.871573,0.943515,266.732725,434.136685,0.952902,312.814813,497.103937,0.933970,282.244873,21.617008,468.370732,25.998887,0.943463,0.007729,,depth_leaf
5,Random Forest,9,"{'n_estimators': 400, 'max_depth': 12, 'min_sa...",268.523595,473.528898,0.943597,265.435375,433.527716,0.953034,312.853033,498.320411,0.933647,282.270668,21.661719,468.459008,26.693335,0.943426,0.007916,,depth_leaf
6,Random Forest,7,"{'n_estimators': 400, 'max_depth': 10, 'min_sa...",270.238772,475.443568,0.943140,266.504952,434.750906,0.952769,310.654712,495.211134,0.934472,282.466145,19.990528,468.468536,25.170725,0.943460,0.007473,,depth_leaf
7,Random Forest,15,"{'n_estimators': 400, 'max_depth': 14, 'min_sa...",268.159234,474.352108,0.943400,267.607665,436.183299,0.952457,312.056312,495.956429,0.934275,282.607737,20.824504,468.830612,24.712641,0.943377,0.007423,,depth_leaf
8,Random Forest,12,"{'n_estimators': 400, 'max_depth': 12, 'min_sa...",270.100096,474.925937,0.943263,268.700785,439.139629,0.951811,312.565322,496.945552,0.934012,283.788734,20.356138,470.337039,23.821204,0.943029,0.007268,,depth_leaf
9,Random Forest,13,"{'n_estimators': 400, 'max_depth': 14, 'min_sa...",269.304453,474.374656,0.943395,267.722030,435.915577,0.952516,314.855971,501.096088,0.932905,283.960818,21.855722,470.462107,26.753265,0.942939,0.008012,,depth_leaf


## Lượt 2 · Thử thêm max_features

Khi cần, đặt `RUN_MAX_FEATURES_CV=True` ở cell Params, chạy lại cell Params và cell bên dưới. Hai bộ dẫn đầu lượt 1 được thử với `max_features=0.6` và `0.8`. Điểm của `1.0` lấy từ lượt 1, nên lượt này có thêm 12 lần fit.

In [16]:
cv_parts = [cv_depth_leaf]
if RUN_MAX_FEATURES_CV:
    top_two = cv_depth_leaf.nsmallest(2, "mean_MAE")
    for params in top_two["params"]:
        extra_results = run_grid_cv(
            raw_train,
            {MODEL_NAME: params},
            {MODEL_NAME: {"max_features": MAX_FEATURES_TO_TRY}},
            n_splits=CV_SPLITS,
        )
        cv_parts.append(extra_results.assign(stage="max_features"))

cv_results = (
    pd.concat(cv_parts, ignore_index=True)
    .sort_values("mean_MAE", kind="stable")
    .reset_index(drop=True)
)
reference = next(
    row for row in cv_depth_leaf.itertuples() if row.params == REFERENCE_PARAMS
)
best = cv_results.iloc[0]
best_params = dict(best["params"])

mae_columns = [f"fold_{fold}_MAE" for fold in range(1, CV_SPLITS + 1)]
mae_columns += ["mean_MAE", "std_MAE"]
comparison = pd.DataFrame(
    [cv_results.iloc[0][mae_columns], pd.Series({key: getattr(reference, key) for key in mae_columns})],
    index=["Bộ tốt nhất", "Bộ đối chiếu"],
)
display(comparison.round(4))
display(cv_results)
print("Tham số tốt nhất:", best_params)

cv_export = cv_results.copy()
cv_export["params"] = cv_export["params"].map(
    lambda params: json.dumps(params, ensure_ascii=False)
)
cv_export.to_csv(OUTPUT_DIR / "grid_cv_results.csv", index=False)
(OUTPUT_DIR / "grid_cv_config.json").write_text(
    json.dumps({
        "n_splits": CV_SPLITS,
        "data_params": DATA_PARAMS,
        "reference_params": REFERENCE_PARAMS,
        "param_grids": PARAM_GRIDS,
        "max_features_cv_enabled": RUN_MAX_FEATURES_CV,
        "max_features_to_try": MAX_FEATURES_TO_TRY,
    }, indent=2),
    encoding="utf-8",
)
(OUTPUT_DIR / "best_params.json").write_text(
    json.dumps(best_params, indent=2), encoding="utf-8",
)

,fold_1_MAE,fold_2_MAE,fold_3_MAE,mean_MAE,std_MAE
Bộ tốt nhất,266.8307,265.3031,311.7181,281.284,21.5292
Bộ đối chiếu,266.8307,265.3031,311.7181,281.284,21.5292


,model,candidate_id,params,fold_1_MAE,fold_1_RMSE,fold_1_R2,fold_2_MAE,fold_2_RMSE,fold_2_R2,fold_3_MAE,fold_3_RMSE,fold_3_R2,mean_MAE,std_MAE,mean_RMSE,std_RMSE,mean_R2,std_R2,warnings,stage
0,Random Forest,10,"{'n_estimators': 400, 'max_depth': 12, 'min_sa...",266.830703,473.474793,0.943610,265.303115,432.351205,0.953289,311.718133,495.621849,0.934363,281.283984,21.529228,467.149282,26.214533,0.943754,0.007727,,depth_leaf
1,Random Forest,6,"{'n_estimators': 400, 'max_depth': 10, 'min_sa...",269.215910,474.390300,0.943391,265.003368,431.665157,0.953437,310.166765,494.298692,0.934713,281.462014,20.370050,466.784716,26.129467,0.943847,0.007651,,depth_leaf
2,Random Forest,5,"{'n_estimators': 400, 'max_depth': 10, 'min_sa...",270.215255,473.645372,0.943569,264.961597,432.312283,0.953297,310.298895,494.878262,0.934560,281.825249,20.247825,466.945306,25.978114,0.943809,0.007651,,depth_leaf
3,Random Forest,11,"{'n_estimators': 400, 'max_depth': 12, 'min_sa...",267.993973,474.336461,0.943404,266.780411,435.121220,0.952688,311.465486,495.290673,0.934451,282.079957,20.784613,468.249451,24.938318,0.943515,0.007446,,depth_leaf
4,Random Forest,14,"{'n_estimators': 400, 'max_depth': 14, 'min_sa...",267.187081,473.871573,0.943515,266.732725,434.136685,0.952902,312.814813,497.103937,0.933970,282.244873,21.617008,468.370732,25.998887,0.943463,0.007729,,depth_leaf
5,Random Forest,9,"{'n_estimators': 400, 'max_depth': 12, 'min_sa...",268.523595,473.528898,0.943597,265.435375,433.527716,0.953034,312.853033,498.320411,0.933647,282.270668,21.661719,468.459008,26.693335,0.943426,0.007916,,depth_leaf
6,Random Forest,7,"{'n_estimators': 400, 'max_depth': 10, 'min_sa...",270.238772,475.443568,0.943140,266.504952,434.750906,0.952769,310.654712,495.211134,0.934472,282.466145,19.990528,468.468536,25.170725,0.943460,0.007473,,depth_leaf
7,Random Forest,15,"{'n_estimators': 400, 'max_depth': 14, 'min_sa...",268.159234,474.352108,0.943400,267.607665,436.183299,0.952457,312.056312,495.956429,0.934275,282.607737,20.824504,468.830612,24.712641,0.943377,0.007423,,depth_leaf
8,Random Forest,12,"{'n_estimators': 400, 'max_depth': 12, 'min_sa...",270.100096,474.925937,0.943263,268.700785,439.139629,0.951811,312.565322,496.945552,0.934012,283.788734,20.356138,470.337039,23.821204,0.943029,0.007268,,depth_leaf
9,Random Forest,13,"{'n_estimators': 400, 'max_depth': 14, 'min_sa...",269.304453,474.374656,0.943395,267.722030,435.915577,0.952516,314.855971,501.096088,0.932905,283.960818,21.855722,470.462107,26.753265,0.942939,0.008012,,depth_leaf


Tham số tốt nhất: {'n_estimators': 400, 'max_depth': 12, 'min_samples_leaf': 5, 'max_features': 1.0, 'random_state': 42, 'n_jobs': -1}


130

## Huấn luyện bộ đã chọn và đánh giá validation

Sau khi xem MAE từng fold, đặt `RUN_VALIDATION=True` trong cell Params, chạy lại cell Params rồi chạy hai cell cuối. So sánh với mốc **240,10 xe/giờ**. Tập test chưa dùng trong thí nghiệm này.

In [17]:
if RUN_VALIDATION:
    from src.experiment_data import load_experiment_splits
    splits = load_experiment_splits(path=DATA_PATH, **DATA_PARAMS, stage="filled")
    featured = {
        name: features.add_weather_features(features.add_time_features(splits[name]))
        for name in ("train", "validation")
    }
    train = featured["train"]
    validation = featured["validation"]
    assert train["date_time"].max() < validation["date_time"].min()

    pipeline = make_config_b_pipeline(MODEL_NAME, best_params)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        start = perf_counter()
        pipeline.fit(train[FEATURE_COLUMNS], train[TARGET])
        fit_seconds = perf_counter() - start

        start = perf_counter()
        raw_prediction = pipeline.predict(validation[FEATURE_COLUMNS])
        predict_seconds = perf_counter() - start

    assert np.isfinite(raw_prediction).all()
    prediction = clip_prediction(raw_prediction)
    metrics = regression_metrics(validation[TARGET], prediction)
    results = pd.DataFrame([{
        "config": CONFIG_NAME,
        "model": MODEL_NAME,
        "split": "validation",
        "n_features": len(FEATURE_COLUMNS),
        "n_train": len(train),
        "n_validation": len(validation),
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        **metrics,
    }])
    predictions = validation[["date_time", TARGET]].copy()
    predictions[f"{MODEL_NAME}__raw"] = raw_prediction
    predictions[MODEL_NAME] = prediction
    warning_table = pd.DataFrame(
        [{"model": MODEL_NAME, "category": w.category.__name__, "message": str(w.message)} for w in caught],
        columns=["model", "category", "message"],
    )

    display(results.round(4))
    if not warning_table.empty:
        display(warning_table)
    print(f"Thay đổi MAE so với ngày 13/11: {metrics['MAE'] - REFERENCE_VALIDATION_MAE:+.2f} xe/giờ")
    results.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
    predictions.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
    warning_table.to_csv(OUTPUT_DIR / "validation_warnings.csv", index=False)
else:
    print("Validation đang tắt. Bật RUN_VALIDATION sau khi xem kết quả CV.")

Validation đang tắt. Bật RUN_VALIDATION sau khi xem kết quả CV.


In [18]:
if RUN_VALIDATION:
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt

    hours_to_plot = 7 * 24
    plot_index = pd.date_range(
        start=predictions["date_time"].min(), periods=hours_to_plot, freq="h",
    )
    plot_data = predictions.set_index("date_time").reindex(plot_index)

    fig, ax = plt.subplots(figsize=(14, 4), layout="constrained")
    ax.plot(plot_index, plot_data[TARGET], color="#202020", label="Thực tế")
    ax.plot(
        plot_index, plot_data[MODEL_NAME], color="tab:orange",
        linestyle="--", label="Dự đoán",
    )
    ax.set_title("Random Forest · Config B · 168 giờ đầu validation")
    ax.set_xlabel("Ngày và giờ")
    ax.set_ylabel("Xe/giờ")
    ax.xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 12]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m\n%H:%M"))
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")

    image_dir = OUTPUT_DIR / "img"
    image_dir.mkdir(exist_ok=True)
    fig.savefig(image_dir / "config_b_random_forest_168h.png", dpi=300, facecolor="white")
    plt.show()

## Nhận xét sau khi chạy

Lần chạy `20261116_214515_288550` đã thử 16 tổ hợp qua 3 fold theo thời gian. Trong lưới này, bộ ngày 13/11 vẫn có MAE CV trung bình thấp nhất là **281,28 xe/giờ**, với 400 cây, `max_depth=12`, `min_samples_leaf=5` và `max_features=1.0`.

| Bộ tham số | Fold 1 | Fold 2 | Fold 3 | MAE CV trung bình |
|---|---:|---:|---:|---:|
| Độ sâu 12, tối thiểu 5 mẫu mỗi lá | **266,83** | 265,30 | 311,72 | **281,28** |
| Độ sâu 10, tối thiểu 5 mẫu mỗi lá | 269,22 | **265,00** | **310,17** | 281,46 |

Giảm độ sâu từ 12 xuống 10 giúp MAE giảm khoảng 0,30 ở fold 2 và 1,55 ở fold 3, nhưng tăng 2,39 ở fold 1. Tính trung bình, bộ độ sâu 10 cao hơn khoảng **0,18 xe/giờ**. Fold 3 có MAE cao nhất ở cả hai bộ. Có thể giữ bộ độ sâu 12 làm mốc và dùng cả hai bộ để thử `max_features=0.6` và `0.8` ở lượt 2.

Lần chạy này mới có kết quả CV. MAE **240,10 xe/giờ** là mốc validation ngày 13/11. Kết quả validation và biểu đồ 168 giờ của lần chạy này sẽ có sau khi bật `RUN_VALIDATION`.

<!-- proposal-assessment -->
## Đánh giá đối với proposal — RQ_2

**Vai trò: HỖ TRỢ.** Mã đối chiếu: [E11](../../docs/experiment_tracking/EXPERIMENTS_BY_DATE_AND_RQ.md). 

- **Câu hỏi:** Bộ tham số RF nào được chọn bằng MAE CV trên train?
- **Đánh giá:** Chọn tham số RF B27 bằng 16 tổ hợp×3 fold trên train, hỗ trợ các ablation weather sau đó. Bộ chọn vẫn là 400 cây, depth12, leaf5, max_features1.0; MAE CV 281,284.
- **Giới hạn kết luận:** Output hiện ghi validation đang tắt. Chỉ báo kết quả CV và tham số chọn; không đếm thành một phép chấm validation mới hoặc kết luận thời tiết có ích.
- **Bằng chứng:** [E11: 20261116_214515_288550](outputs/20261116_214515_288550).

Đánh giá trên dựa vào hiện vật lịch sử đã đối chứng, không phải kết quả chạy mới. Code lấy dữ liệu đã được chuẩn hóa; chưa huấn luyện lại hoặc chấm test.
